# CS 363M Final Project: Social Media Bot Detection

## Team

- Rohan Yelandur
- Odin Limaye
- Eesha Kandukuri
- Myshae Allen

## Problem Statement

Detect text-based social media posts that have been uploaded by bots.

## Dataset Description

Our dataset is called the "Fake Social Media Account Detection Dataset" which contains social media profiles from across FaceBook, Twitter, and Instagram.

Data contains information about follower patterns and profile characteristics.

We are using two specific datasets, real_users.csv and fake_users.csv (can be found at links below):

  - https://www.kaggle.com/datasets/satishkumarprajapati/fake-social-media-account-detection-dataset?select=fake_users.csv
  - https://www.kaggle.com/datasets/satishkumarprajapati/fake-social-media-account-detection-dataset?select=real_users.csv


In [1]:
import pandas as pd

In [2]:
# Dataset Brief View (Exploration):
data_fake_users = pd.read_csv("data/fake_users.csv")
data_fake_users["is_fake"] = 1
data_real_users = pd.read_csv("data/real_users.csv")
data_real_users["is_fake"] = 0
df = pd.concat([data_fake_users, data_real_users])

print(f' Combined Datasets Shape: {df.shape}')
display(df.head())


 Combined Datasets Shape: (5000, 35)


,id,name,screen_name,statuses_count,followers_count,friends_count,favourites_count,listed_count,created_at,url,...,profile_background_image_url,profile_background_color,profile_link_color,utc_offset,protected,verified,description,updated,dataset,is_fake
0,1.664073e+09,Lavona Keller,kellervxg,13.093733,-0.000000,0.000000,0.0,0.0,Sun Jun 24 03:26:11 +0000 2012,-0.0,...,NaN,C0DEED,NaN,0.0,0.0,0.0,Keep moving forwad,2015-02-14 10:40:01,NaN,1
1,7.296345e+08,Elizabet Graves,NaN,0.000000,0.000000,0.000000,0.0,0.0,Mon Jun 25 02:06:59 +0000 2012,0.0,...,http://a0.twimg.com/images/themes/theme1/bg.png,NaN,0084B4,0.0,0.0,0.0,NaN,2015-02-14 10:40:01,NaN,1
2,1.214885e+09,Estelle Santos,estellesant,52.941642,-0.000000,518.508286,-0.0,-0.0,Sat Jun 23 19:48:28 +0000 2012,-0.0,...,NaN,C0DEED,0084B4,0.0,0.0,0.0,"Practical wisdom,healh, Chinese Medicine",NaN,NaN,1
3,-0.000000e+00,Patsy Weiss,patsyyor,18.857321,0.000000,1059.138617,0.0,0.0,Sat Jun 23 15:34:23 +0000 2012,0.0,...,http://a0.twimg.com/images/themes/theme1/bg.png,C0DEED,0084B4,0.0,0.0,0.0,this is a support page or el_aminito check him...,NaN,INT,1
4,1.844412e+09,Cindy Cook,NaN,51.167110,7.368223,0.000000,0.0,0.0,NaN,0.0,...,http://a0.twimg.com/images/themes/theme1/bg.png,NaN,NaN,0.0,0.0,-0.0,NaN,NaN,NaN,1


# Data Cleaning

In [3]:
# Data Cleaning:
print('Counts of features with missing values:')
print(df.isna().sum().loc[lambda x: x > 0].head(10))

cols_that_should_be_int = ['statuses_count', 'followers_count', 'friends_count', 'favourites_count', 
                           'listed_count']
print('\nCounts of integer features that are decimal:')
print((df[cols_that_should_be_int] % 1 != 0).sum().head(10))

cols_that_should_be_binary = ['default_profile', 'default_profile_image', 'geo_enabled',
             'profile_use_background_image', 'profile_background_tile']
print('\nCounts of binary features that are not 0 or 1:')
print((~df[cols_that_should_be_binary].isin([0, 1])).sum().head(10))

# BEFORE changing cols_that_should_be_binary into binary
print("\nCcols that should be binary (BEFORE conversion):")
len_of_longest_col_name = max(len(c) for c in cols_that_should_be_binary)
for col in cols_that_should_be_binary:
    print(col," before:\n", df[col].head(5), "\n", sep="")

# AFTER changing cols_that_should_be_binary into binary
print("\nCols that should be binary (AFTER conversion):")
for col in cols_that_should_be_binary:
    df[col] = (df[col] > 0).astype(int)
for col in cols_that_should_be_binary:
    print(col, "after:\n", df[col].head(5), "\n", sep="")


Counts of features with missing values:
name                                  1740
screen_name                           1770
created_at                            1739
url                                   1986
lang                                  1770
time_zone                             3879
location                              2319
profile_image_url                     1789
profile_banner_url                    3922
profile_background_image_url_https    1731
dtype: int64

Counts of integer features that are decimal:
statuses_count      2396
followers_count     2329
friends_count       2475
favourites_count    1109
listed_count         640
dtype: int64

Counts of binary features that are not 0 or 1:
default_profile                 1540
default_profile_image              7
geo_enabled                      566
profile_use_background_image    2355
profile_background_tile          403
dtype: int64

Ccols that should be binary (BEFORE conversion):
default_profile before:
0    2.874910

In [ ]:
# drop the unneeded features:
df = df.drop(columns=['id', 'dataset', 'updated', 'verified', 'protected', 'time_zone'])

# fix -0.0 values in every float column:
float_cols = df.select_dtypes(include="float").columns
df[float_cols] = df[float_cols] + 0.0 # adding 0.0 turns -0.0 into a plain 0.0

# fix url values to be 0 or 1 as well (handled differently because these are strings)
def url_to_binary(x):
    if pd.isna(x):
        return 0
    if isinstance(x, (int, float)):
        return 0 if x == 0 else 1
    s = str(x).strip()
    if s == '' or s in ('-0.0', '0', '0.0'):
        return 0
    return 1
df['url'] = df['url'].map(url_to_binary).astype(int)

# binary encode profile_banner_url (1 = has a banner, 0 = no banner)
df['has_profile_banner'] = df['profile_banner_url'].map(url_to_binary).astype(int)
df = df.drop(columns=['profile_banner_url'])

# make all floats to be truncated into ints (except for utc_offset)
# uses cols_that_should_be_int from previous cell
df[cols_that_should_be_int] = df[cols_that_should_be_int].fillna(0).astype(int)

# make all NaN's int 0's as well
df = df.fillna(0)

# still need to one-hot encode colors...
df = pd.get_dummies(df, columns=["profile_sidebar_border_color", "profile_sidebar_fill_color", "profile_use_background_image", "profile_background_image_url_https"])
display(df.head())

#remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)
print('Shape after removing duplicates:', df.shape)

# still need to conside if there were any columns that were supposed to be one-hot-encoded that weren't?...



,name,screen_name,statuses_count,followers_count,friends_count,favourites_count,listed_count,created_at,url,lang,...,profile_background_image_url_https_https://twimg0-a.akamaihd.net/profile_background_images/740580425/59f5f29047cd16579375c3d04bb43227.jpeg,profile_background_image_url_https_https://twimg0-a.akamaihd.net/profile_background_images/753174473/038c6df13483fc96519989f178d6bb1d.jpeg,profile_background_image_url_https_https://twimg0-a.akamaihd.net/profile_background_images/753793709/c834c9322bbe0736dbec49a92a0f1baf.jpeg,profile_background_image_url_https_https://twimg0-a.akamaihd.net/profile_background_images/770068140/f9fc87593d52e757ce86478b78580894.jpeg,profile_background_image_url_https_https://twimg0-a.akamaihd.net/profile_background_images/770120781/f6dac170ccb561cc92e02baaba589a8f.jpeg,profile_background_image_url_https_https://twimg0-a.akamaihd.net/profile_background_images/796518562/72f5a88eee3a9e98917134018f507b5b.png,profile_background_image_url_https_https://twimg0-a.akamaihd.net/profile_background_images/800338070/efe46962f13c62ccfbac45f9653cdab0.png,profile_background_image_url_https_https://twimg0-a.akamaihd.net/profile_background_images/803375286/ea69079cac1ab4270c2fd34f993da3b0.jpeg,profile_background_image_url_https_https://twimg0-a.akamaihd.net/profile_background_images/805444976/8795ef7f04fe31ea71a603f88baa606c.jpeg,profile_background_image_url_https_https://twimg0-a.akamaihd.net/profile_background_images/807849160/19c58607f0a76b7871b53128982ee314.png
0,Lavona Keller,kellervxg,13,0,0,0,0,Sun Jun 24 03:26:11 +0000 2012,0,en,...,False,False,False,False,False,False,False,False,False,False
1,Elizabet Graves,0,0,0,0,0,0,Mon Jun 25 02:06:59 +0000 2012,0,en,...,False,False,False,False,False,False,False,False,False,False
2,Estelle Santos,estellesant,52,0,518,0,0,Sat Jun 23 19:48:28 +0000 2012,0,en,...,False,False,False,False,False,False,False,False,False,False
3,Patsy Weiss,patsyyor,18,0,1059,0,0,Sat Jun 23 15:34:23 +0000 2012,0,en,...,False,False,False,False,False,False,False,False,False,False
4,Cindy Cook,0,51,7,0,0,0,0,0,en,...,False,False,False,False,False,False,False,False,False,False


# Data Exploration & Preprocessing
- Show the code, data exploration outputs, and visualizations. Display what you cleaned and engineered. Include visualizations of data characteristics.


In [5]:
# Before exploring data, just quickly displaying the new current state of data after cleaning... (data exploration)
print('df shape: ', df.shape)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
display(df.head())

df shape:  (5000, 775)


,name,screen_name,statuses_count,followers_count,friends_count,favourites_count,listed_count,created_at,url,lang,location,default_profile,default_profile_image,geo_enabled,profile_image_url,profile_text_color,profile_image_url_https,profile_background_tile,profile_background_image_url,profile_background_color,profile_link_color,utc_offset,description,is_fake,has_profile_banner,profile_sidebar_border_color_0,profile_sidebar_border_color_000000,profile_sidebar_border_color_0077B1,profile_sidebar_border_color_050D2E,profile_sidebar_border_color_051733,profile_sidebar_border_color_070DB0,profile_sidebar_border_color_08C20E,profile_sidebar_border_color_090A0A,profile_sidebar_border_color_0D0D0C,profile_sidebar_border_color_0D0D0D,profile_sidebar_border_color_11A1D6,profile_sidebar_border_color_140202,profile_sidebar_border_color_181A1E,profile_sidebar_border_color_182952,profile_sidebar_border_color_1E3157,profile_sidebar_border_color_1F0909,profile_sidebar_border_color_314163,profile_sidebar_border_color_373737,profile_sidebar_border_color_390402,profile_sidebar_border_color_404040,profile_sidebar_border_color_417E20,profile_sidebar_border_color_4392BA,profile_sidebar_border_color_464E52,profile_sidebar_border_color_4A2561,profile_sidebar_border_color_4A66C1,profile_sidebar_border_color_4B9C82,profile_sidebar_border_color_55D0F5,profile_sidebar_border_color_5D0B7D,profile_sidebar_border_color_5ED4DC,profile_sidebar_border_color_5F9986,profile_sidebar_border_color_65B0DA,profile_sidebar_border_color_6BCBFF,profile_sidebar_border_color_6D7011,profile_sidebar_border_color_763EDA,profile_sidebar_border_color_794725,profile_sidebar_border_color_7A5C45,profile_sidebar_border_color_7D8A3A,profile_sidebar_border_color_8024BD,profile_sidebar_border_color_829D5E,profile_sidebar_border_color_86A4A6,profile_sidebar_border_color_999999,profile_sidebar_border_color_9DB98C,profile_sidebar_border_color_9E0C0C,profile_sidebar_border_color_9E712E,profile_sidebar_border_color_A0EEF5,profile_sidebar_border_color_A3BECC,profile_sidebar_border_color_A66F0C,profile_sidebar_border_color_A8C7F7,profile_sidebar_border_color_ADF1FC,profile_sidebar_border_color_B3BA9B,profile_sidebar_border_color_B68B9E,profile_sidebar_border_color_B9E5CE,profile_sidebar_border_color_BCA3D9,profile_sidebar_border_color_BCB302,profile_sidebar_border_color_BDDCAD,profile_sidebar_border_color_C0DCF1,profile_sidebar_border_color_C0DEED,profile_sidebar_border_color_C296A5,profile_sidebar_border_color_C2C1E6,profile_sidebar_border_color_C2C5D1,profile_sidebar_border_color_C4CAB0,profile_sidebar_border_color_C5DEE0,profile_sidebar_border_color_C6E2EE,profile_sidebar_border_color_CC3366,profile_sidebar_border_color_D3D2CF,profile_sidebar_border_color_D40F51,profile_sidebar_border_color_D4C6AC,profile_sidebar_border_color_D6F3FF,profile_sidebar_border_color_D9B17E,profile_sidebar_border_color_DADEDE,profile_sidebar_border_color_DBE9ED,profile_sidebar_border_color_DFDFDF,profile_sidebar_border_color_E1E1E1,profile_sidebar_border_color_E2EAEF,profile_sidebar_border_color_E62910,profile_sidebar_border_color_EEEEEE,profile_sidebar_border_color_F0A830,profile_sidebar_border_color_F1F2E8,profile_sidebar_border_color_F2E195,profile_sidebar_border_color_F52739,profile_sidebar_border_color_F9C5A3,profile_sidebar_border_color_FA8100,profile_sidebar_border_color_FAFAFA,profile_sidebar_border_color_FCFCFC,profile_sidebar_border_color_FF0000,profile_sidebar_border_color_FF0F23,profile_sidebar_border_color_FFAAAA,profile_sidebar_border_color_FFF8AD,profile_sidebar_border_color_FFFE1A,profile_sidebar_border_color_FFFFFF,profile_sidebar_border_color_a8c7f7,profile_sidebar_border_color_eeeeee,profile_sidebar_fill_color_0,profile_sidebar_fill_color_000000,profile_sidebar_fill_color_000B17,profile_sidebar_fill_color_00E1FF,profile_sidebar_fill_color_030516,profile_sidebar_fill_color_051F1B,profile_sidebar_fill_color_080808,profile_sidebar_fill_color_0A061A,profile_sidebar_fill_color_0E0E0E,profile_side

# Modeling
- Display your model implementations, code, and results. Show how different algorithms performed. Include relevant outputs and comparisons.


In [6]:
# Code here (Modeling)...

# Rohan (Neural Nets)

# Myshae (Decision Trees)




# Results & Analysis
- Show model outputs, performance metrics, visualizations of results, error analysis, and insights. Interpret what you found.



In [7]:
# Code here (Results a& Analysis)...
